# Inspect Model Predictions

Manually inspect model tool call predictions against ground truth.
- View the input context fed to the model
- Compare predicted vs gold tool calls
- Filter by correct/incorrect, category, model

In [ ]:
import json
import re
from pathlib import Path
from IPython.display import display, HTML

OUTPUT_DIR = Path("output")
GOLD_PATH = Path("data/hermes_reasoning_tool_use_test_split_tool_calls_only.jsonl")

# --- Tool call parsing (same as eval_tool_calls.py) ---

def parse_last_tool_call(content):
    """Find the last <tool_call>...</tool_call> and parse the JSON inside."""
    close_pos = content.rfind("</tool_call>")
    if close_pos == -1:
        return None
    open_pos = content.rfind("<tool_call>", 0, close_pos)
    if open_pos == -1:
        return None
    raw = content[open_pos + len("<tool_call>"):close_pos].strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        if start == -1:
            return None
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == "{": depth += 1
            elif raw[i] == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(raw[start:i+1])
                    except json.JSONDecodeError:
                        return None
    return None

def normalize_value(v):
    if isinstance(v, str):
        try: return int(v)
        except ValueError:
            try: return float(v)
            except ValueError: return v.strip()
    if isinstance(v, dict): return normalize_args(v)
    if isinstance(v, list): return [normalize_value(x) for x in v]
    return v

def normalize_args(args):
    if isinstance(args, str):
        try: args = json.loads(args)
        except json.JSONDecodeError: return {"_raw": args.strip()}
    if not isinstance(args, dict): return {"_raw": str(args)}
    return {k: normalize_value(v) for k, v in sorted(args.items())}

def sanitize(text):
    return re.sub(r'[\ud800-\udfff]', '', str(text))

print("Utilities loaded.")

In [ ]:
# Load gold data
gold_records = []
with open(GOLD_PATH) as f:
    for line in f:
        gold_records.append(json.loads(line))

# Discover prediction files
pred_files = sorted(OUTPUT_DIR.glob("*_preds.jsonl"))
model_names = [f.stem.replace("_preds", "") for f in pred_files]

print(f"Gold examples: {len(gold_records)}")
print(f"Models found: {model_names}")

# Load all predictions
all_preds = {}
for name, path in zip(model_names, pred_files):
    with open(path) as f:
        preds = [json.loads(line) for line in f]
    all_preds[name] = preds
    print(f"  {name}: {len(preds)} predictions")

In [ ]:
# Build comparison table for each model
comparisons = {}  # model_name -> list of dicts

for model_name, preds in all_preds.items():
    rows = []
    for i, (gold, pred) in enumerate(zip(gold_records, preds)):
        gold_tc = gold["metadata"]["gold_tool_call"]
        gold_asst = gold["metadata"].get("gold_assistant_content", "")
        pred_content = pred["content"]
        pred_tc = parse_last_tool_call(pred_content)

        if pred_tc is None:
            name_ok, args_ok, exact = False, False, False
        else:
            name_ok = pred_tc.get("name") == gold_tc.get("name")
            args_ok = normalize_args(pred_tc.get("arguments", {})) == normalize_args(gold_tc.get("arguments", {}))
            exact = name_ok and args_ok

        rows.append({
            "index": i,
            "category": gold["metadata"]["scenario_category"],
            "gold_tc": gold_tc,
            "gold_assistant_content": gold_asst,
            "pred_tc": pred_tc,
            "pred_content": pred_content,
            "input_messages": gold["messages"],
            "name_ok": name_ok,
            "args_ok": args_ok,
            "exact": exact,
            "parse_fail": pred_tc is None,
        })
    comparisons[model_name] = rows

print("Comparisons built for all models.")
for name, rows in comparisons.items():
    exact = sum(r["exact"] for r in rows)
    fails = sum(r["parse_fail"] for r in rows)
    print(f"  {name}: {exact}/{len(rows)} exact match, {fails} parse failures")

In [ ]:
ROLE_COLORS = {
    "system": "#6c757d",
    "user": "#0d6efd",
    "assistant": "#198754",
    "tool": "#dc3545",
}


def fmt(text, max_len=1500):
    """Sanitize and truncate text for HTML display."""
    text = sanitize(text)
    if len(text) > max_len:
        text = text[:max_len] + "\n... [truncated]"
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def display_example(row, model_name):
    """Display a single prediction with input context, gold, and predicted tool call."""
    status_color = "#198754" if row["exact"] else ("#fd7e14" if row["name_ok"] else "#dc3545")
    if row["parse_fail"]:
        status = "PARSE FAIL"
    elif row["exact"]:
        status = "EXACT MATCH"
    elif row["name_ok"]:
        status = "NAME OK, ARGS WRONG"
    else:
        status = "WRONG"

    html = f"<div style='border:2px solid {status_color}; border-radius:8px; padding:12px; margin:10px 0;'>"

    # Header
    html += (f"<div style='margin-bottom:10px;'>"
             f"<b>Example {row['index']}</b> | "
             f"<code>{row['category']}</code> | "
             f"Model: <b>{model_name}</b> | "
             f"<span style='color:{status_color}; font-weight:bold;'>{status}</span>"
             f"</div>")

    # Input context (collapsible)
    html += "<details><summary style='cursor:pointer; font-weight:bold; margin:6px 0;'>Input Context (click to expand)</summary>"
    for msg in row["input_messages"]:
        role = msg["role"]
        color = ROLE_COLORS.get(role, "#333")
        html += (f"<div style='margin:4px 0; padding:6px; "
                 f"border-left:3px solid {color}; background:#f8f9fa; font-size:12px;'>"
                 f"<b style='color:{color};'>{role.upper()}</b>"
                 f"<pre style='white-space:pre-wrap; margin:4px 0 0 0; font-size:12px;'>{fmt(msg['content'])}</pre>"
                 f"</div>")
    html += "</details>"

    # Gold vs Predicted tool call (side by side)
    gold_json = json.dumps(row["gold_tc"], indent=2)
    pred_json = json.dumps(row["pred_tc"], indent=2) if row["pred_tc"] else "[no tool call parsed]"

    html += "<div style='display:flex; gap:12px; margin:8px 0;'>"

    # Gold tool call
    html += (f"<div style='flex:1; padding:8px; background:#d4edda; border-radius:6px;'>"
             f"<b>Gold Tool Call</b>"
             f"<pre style='white-space:pre-wrap; margin:4px 0 0 0; font-size:13px;'>{fmt(gold_json, 1000)}</pre>"
             f"</div>")

    # Predicted tool call
    pred_bg = "#d4edda" if row["exact"] else ("#fff3cd" if row["name_ok"] else "#f8d7da")
    html += (f"<div style='flex:1; padding:8px; background:{pred_bg}; border-radius:6px;'>"
             f"<b>Predicted Tool Call</b>"
             f"<pre style='white-space:pre-wrap; margin:4px 0 0 0; font-size:13px;'>{fmt(pred_json, 1000)}</pre>"
             f"</div>")

    html += "</div>"

    # Gold vs Predicted full assistant response (side by side)
    gold_asst = row.get("gold_assistant_content", "")
    pred_asst = row["pred_content"]

    html += "<details open><summary style='cursor:pointer; font-weight:bold; margin:6px 0;'>Gold vs Predicted Full Response</summary>"
    html += "<div style='display:flex; gap:12px; margin:4px 0;'>"

    html += (f"<div style='flex:1; padding:8px; background:#d4edda; border-radius:6px;'>"
             f"<b>Gold Assistant Response</b>"
             f"<pre style='white-space:pre-wrap; margin:4px 0 0 0; font-size:12px;'>{fmt(gold_asst, 2000)}</pre>"
             f"</div>")

    html += (f"<div style='flex:1; padding:8px; background:{pred_bg}; border-radius:6px;'>"
             f"<b>Predicted Response</b>"
             f"<pre style='white-space:pre-wrap; margin:4px 0 0 0; font-size:12px;'>{fmt(pred_asst, 2000)}</pre>"
             f"</div>")

    html += "</div></details>"

    html += "</div>"
    display(HTML(html))

print("Display function ready.")

## Pick a model and browse examples

In [ ]:
# === CONFIGURE HERE ===
MODEL = "llama3.1_8b"     # Change to any model name from the list above
START = 0                  # Starting index
COUNT = 5                  # Number of examples to show
# ======================

rows = comparisons[MODEL]
for row in rows[START:START+COUNT]:
    display_example(row, MODEL)

## Filter: show only failures

In [ ]:
MODEL = "llama3.1_8b"
COUNT = 5

failures = [r for r in comparisons[MODEL] if not r["exact"]]
print(f"{len(failures)} failures out of {len(comparisons[MODEL])} total")

for row in failures[:COUNT]:
    display_example(row, MODEL)

## Filter: show only parse failures

In [ ]:
MODEL = "smollm2_135m"
COUNT = 5

parse_fails = [r for r in comparisons[MODEL] if r["parse_fail"]]
print(f"{len(parse_fails)} parse failures out of {len(comparisons[MODEL])} total")

for row in parse_fails[:COUNT]:
    display_example(row, MODEL)

## Filter by scenario category

In [ ]:
MODEL = "llama3.1_8b"
CATEGORY = "multistep"  # single, multiturn, multistep
COUNT = 5

filtered = [r for r in comparisons[MODEL] if r["category"] == CATEGORY]
exact = sum(r["exact"] for r in filtered)
print(f"{CATEGORY}: {exact}/{len(filtered)} exact match")

for row in filtered[:COUNT]:
    display_example(row, MODEL)

## Compare models side-by-side on same example

In [ ]:
IDX = 0  # Example index to compare

for model_name in comparisons:
    display_example(comparisons[model_name][IDX], model_name)

## Name-correct but args-wrong examples

In [ ]:
MODEL = "llama3.1_8b"
COUNT = 5

name_ok_args_wrong = [r for r in comparisons[MODEL] if r["name_ok"] and not r["args_ok"]]
print(f"{len(name_ok_args_wrong)} examples with correct name but wrong args")

for row in name_ok_args_wrong[:COUNT]:
    display_example(row, MODEL)